# 01 — Language Coverage Exploration

Explores `language_codes_comprehensive.csv` to understand the full language target set.

The dataset combines five sources into ~880 rows (special-purpose non-language codes like `und` and `zxx` are excluded at build time):

| Source | Role | ~Count | `cldr_version` |
|--------|------|--------|----------------|
| **CLDR 48.2** (npm `cldr-core` + `cldr-localenames-full`) | Scripts, directionality, modern-use flag, official status | ~802 codes | `'48.2.0'` |
| **LOC ISO 639-2** | ISO 639-1/2 code mappings for individual languages | ~190 codes | `''` |
| **Wikimedia** | Languages with active Wikipedia projects | ~270 codes | `''` |
| **CLDR v45 supplement** | ISO 639-3 languages in CLDR 45 but dropped from 48.2 (contributor policy, not linguistics) | 23 codes | `'45'` |
| **SIL ISO 639-3** | English reference names for ~190 CLDR codes absent from CLDR's `en/languages.json` — name-enrichment only, adds no new language codes | — | — |

Family hierarchy (§1.3) also comes from CLDR's `languageGroups.json`, part of the same `cldr-core` package. Both npm packages are downloaded from the npm registry and cached locally. Cite as: *Unicode CLDR version 48.2.0, https://github.com/unicode-org/cldr-json.*

All 880 codes in the output CSV have a non-empty value for every key column: `language_code`, `language_name` (English), `family_name`, `directionality`, and `sources`.

Sections:
- **§1.1 Source Coverage and Overlap** — how many languages come from each source, how sources overlap, CLDR version provenance, and SIL English name enrichment
- **§1.2 Constructing the Target Set** — why ~880? filter strategies, expansion steps, non-modern languages, and directionality overrides
- **§1.3 Language Families** — family distribution, family sizes, and how family assignments were made
- **§1.4 Script and Directionality** — script landscape, LTR/RTL split, directionality by family and source

In [1]:
import os
import sys
import pandas as pd
import altair as alt
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))
from scripts.utils import get_data_directory_path

DATA_DIR = get_data_directory_path()
CSV_PATH = os.path.join(DATA_DIR, "metadata_files", "language_codes_comprehensive.csv")

# converters preserves 'nan' (Min Nan Chinese, ISO 639-3) as a string
df = pd.read_csv(CSV_PATH, converters={'language_code': str})

# n_languages is the single source of truth for the target set size used throughout this notebook
n_languages = df["language_code"].nunique()
print(f"Total languages: {n_languages} unique codes ({len(df)} rows)")
print(f"Columns: {list(df.columns)}")

Retrieving translation pipeline data directory path...

Total languages: 880 unique codes (880 rows)
Columns: ['language_code', 'language_name', 'local_name', 'iso639_1', 'iso639_2_t', 'iso639_2_b', 'scripts', 'script_codes', 'primary_script', 'primary_script_code', 'directionality', 'directionality_wikimedia', 'modern_language', 'is_official', 'n_scripts', 'in_iso639_1', 'in_iso639_2', 'in_wikimedia', 'is_group_iso639_2', 'sources', 'cldr_version', 'iso639_5_direct', 'iso639_5_family', 'family_name', 'subfamily_name']


In [2]:
import json
from pathlib import Path
from scripts.experiment.generate_language_codes import MANUAL_LANG_TO_SET5

_json_path = Path("..") / "datasets" / "metadata_files" / "language_family_assignments.json"
with open(_json_path, encoding="utf-8") as _f:
    _json_entries = {e["language_code"]: e for e in json.load(_f)}

def _family_source(row):
    raw_code = row["language_code"]
    # "nan" is the ISO 639-3 code for Min Nan Chinese. Without converters={'language_code': str},
    # pd.read_csv promotes the string "nan" to float NaN. Normalise it back so lookups work.
    code = "nan" if pd.isna(raw_code) else str(raw_code).strip()
    iso2 = str(row.get("iso639_2_t", "") or "").strip()
    fname = row["family_name"]
    if pd.isna(fname) or str(fname).strip() == "":
        return "Unassigned"
    if MANUAL_LANG_TO_SET5.get(code) or MANUAL_LANG_TO_SET5.get(iso2):
        return "Manual mapping"
    if code in _json_entries:
        return "JSON assessment"
    return "Unassigned"

df["family_name_source"] = df.apply(_family_source, axis=1)
print(df["family_name_source"].value_counts().to_string())

family_name_source
Manual mapping     664
JSON assessment    216


/Users/zleblanc/.virtualenvs/spring-2026-env/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [3]:
# Show the languages that are unassigned to a family to double-check if there are any obvious candidates for manual assignment or JSON assessment.
df[df.family_name_source == "Unassigned"]

,language_code,language_name,local_name,iso639_1,iso639_2_t,iso639_2_b,scripts,script_codes,primary_script,primary_script_code,...,in_iso639_2,in_wikimedia,is_group_iso639_2,sources,cldr_version,iso639_5_direct,iso639_5_family,family_name,subfamily_name,family_name_source


In [4]:
# Show the original translated DH terms dataset to get a sense of the number of translations and languages covered. This dataset was generated in December 2022.
original_translated_dh_terms = pd.read_csv(os.path.join("..", "historic_materials", "translated_dh_terms.csv"))
original_translated_dh_terms = original_translated_dh_terms[original_translated_dh_terms.term_source == "Digital Humanities"]
print(f"Original translated DH terms: {len(original_translated_dh_terms)} rows")
print(f"Number of actually translated terms: {original_translated_dh_terms[original_translated_dh_terms.translated_term.notnull()].shape[0]} unique translations")
print(f"{original_translated_dh_terms[original_translated_dh_terms.translated_term.isnull()].language_name.nunique()} Languages not translated: {original_translated_dh_terms[original_translated_dh_terms.translated_term.isnull()].language_name.unique().tolist()}")

Original translated DH terms: 185 rows
Number of actually translated terms: 123 unique translations
62 Languages not translated: ['Abkhaz', 'Afar', 'Aragonese', 'Avaric', 'Avestan', 'Bashkir', 'Bislama', 'Breton', 'Chamorro', 'Chechen', 'Chuvash', 'Cornish', 'Cree', 'Dzongkha', 'Faroese', 'Fijian', 'Fula', 'Herero', 'Hiri Motu', 'Interlingua', 'Interlingue', 'Inupiaq', 'Ido', 'Inuktitut', 'Kalaallisut', 'Kanuri', 'Kashmiri', 'Kikuyu, Gikuyu', 'Komi', 'Kongo', 'Kwanyama, Kuanyama', 'Limburgish', 'Luba-Katanga', 'Manx', 'Marshallese', 'Nauru', 'Navajo, Navaho', 'North Ndebele', 'Ndonga', 'Norwegian Nynorsk', 'Nuosu', 'South Ndebele', 'Occitan', 'Ojibwe, Ojibwa', 'Old Church Slavonic', 'Ossetian, Ossetic', 'Pāli', 'Romansh', 'Kirundi', 'Sardinian', 'Northern Sami', 'Sango', 'Swati', 'Tibetan', 'Tswana', 'Tonga', 'Tahitian', 'Venda', 'Volapük', 'Walloon', 'Wolof', 'Zhuang, Chuang']


## 1.1 Source Coverage and Overlap

How many languages come from each source, and how much do they overlap?

Each source covers a different slice of the 880 languages, and many languages appear in more than one. The first chart shows raw counts per source; the second shows the specific source combinations (CLDR only, CLDR + Wikimedia, all three, etc.).

In [5]:
source_flags = {
    "CLDR":      df["sources"].str.contains("cldr", na=False),
    "LOC ISO 639-2": df["in_iso639_2"].fillna(False).astype(bool),
    "Wikimedia": df["in_wikimedia"].fillna(False).astype(bool),
}

summary = pd.DataFrame({
    "source": list(source_flags.keys()),
    "count":  [m.sum() for m in source_flags.values()],
})

bars = alt.Chart(summary).mark_bar().encode(
    y=alt.Y("source:N", sort="-x", title=None),
    x=alt.X("count:Q", title="number of language codes"),
    color=alt.Color("source:N", legend=None, scale=alt.Scale(scheme="tableau10")),
    tooltip=["source:N", "count:Q"],
).properties(width=380, height=120, title="Language codes per source")

text = bars.mark_text(align="left", dx=4, fontSize=10).encode(
    text="count:Q", color=alt.value("black"))

(bars + text) 

alt.LayerChart(...)

In [6]:
# ── Source combination breakdown ─────────────────────────────────────────────
combo_rows = []
for _, row in df.iterrows():
    in_cldr = "cldr" in str(row.get("sources", "") or "")
    in_iso2 = bool(row.get("in_iso639_2"))
    in_wiki = bool(row.get("in_wikimedia"))
    parts = []
    if in_cldr: parts.append("CLDR")
    if in_iso2:  parts.append("ISO 639-2")
    if in_wiki:  parts.append("Wikimedia")
    combo_rows.append(" + ".join(parts) if parts else "None")

df["source_combo"] = combo_rows
combo_counts = df["source_combo"].value_counts().reset_index()
combo_counts.columns = ["combination", "count"]
combo_counts["n_sources"] = combo_counts["combination"].apply(
    lambda c: 0 if c == "None" else len(c.split(" + "))
)
print(combo_counts.to_string(index=False))

_venn_csv = os.path.join(DATA_DIR, "metadata_files", "source_overlap_venn.csv")
combo_counts[["combination", "count"]].to_csv(_venn_csv, index=False)
print(f"\nSaved for RAW Graphs → {_venn_csv}")

# ── Bubble plot ───────────────────────────────────────────────────────────────
# x = number of databases in agreement — a genuinely meaningful axis.
# y = manual jitter to separate bubbles within each column.
# Area ∝ language count; labels printed inside large bubbles, below small ones.
# Single-source colors match the established source palette; multi-source
# bubbles use blended hues (blue+orange→purple, blue+green→teal, etc.).
_COMBO_COLORS = {
    "CLDR":                         "#1976d2",
    "ISO 639-2":                    "#e64a19",
    "Wikimedia":                    "#388e3c",
    "CLDR + ISO 639-2":             "#7040a0",
    "CLDR + Wikimedia":             "#1a8060",
    "ISO 639-2 + Wikimedia":        "#b06820",
    "CLDR + ISO 639-2 + Wikimedia": "#444444",
}

_BUBBLE_POS = {
    "CLDR":                         (1,  0.55),
    "Wikimedia":                    (1,  0.0),
    "ISO 639-2":                    (1, -0.45),
    "CLDR + ISO 639-2":             (2,  0.35),
    "CLDR + Wikimedia":             (2,  0.0),
    "ISO 639-2 + Wikimedia":        (2, -0.30),
    "CLDR + ISO 639-2 + Wikimedia": (3,  0.0),
}

bubble_df = combo_counts.copy()
bubble_df["x"] = bubble_df["combination"].map(lambda c: _BUBBLE_POS[c][0])
bubble_df["y"] = bubble_df["combination"].map(lambda c: _BUBBLE_POS[c][1])

_xscale = alt.Scale(domain=[0.3, 3.7])
_yscale = alt.Scale(domain=[-0.75, 0.85])

bubbles = alt.Chart(bubble_df).mark_circle(opacity=0.88).encode(
    x=alt.X("x:Q", title="source databases in agreement",
            scale=_xscale,
            axis=alt.Axis(values=[1, 2, 3], grid=True, gridDash=[4, 4])),
    y=alt.Y("y:Q", axis=None, scale=_yscale),
    size=alt.Size("count:Q",
                  scale=alt.Scale(type="linear", range=[250, 9000]),
                  legend=None),
    color=alt.Color("combination:N",
                    scale=alt.Scale(
                        domain=list(_COMBO_COLORS.keys()),
                        range=list(_COMBO_COLORS.values()),
                    ),
                    legend=alt.Legend(title="combination")),
    tooltip=["combination:N", "count:Q"],
)

# Count labels: inside (white) for large bubbles, below (dark) for small ones
_big   = bubble_df[bubble_df["count"] > 20].copy()
_small = bubble_df[bubble_df["count"] <= 20].copy()

big_labels = alt.Chart(_big).mark_text(
    color="white", fontWeight="bold", fontSize=11,
    align="center", baseline="middle",
).encode(
    x=alt.X("x:Q", scale=_xscale),
    y=alt.Y("y:Q", scale=_yscale),
    text="count:Q",
)

small_labels = alt.Chart(_small).mark_text(
    color="#333", fontWeight="bold", fontSize=10,
    align="center", dy=22,
).encode(
    x=alt.X("x:Q", scale=_xscale),
    y=alt.Y("y:Q", scale=_yscale),
    text="count:Q",
)

(bubbles + big_labels + small_labels).properties(
    width=480, height=340,
    title=alt.Title(
        "Source consensus across 880 languages",
        subtitle="Bubble area ∝ language count  ·  x = how many databases include these languages",
    ),
).configure_view(strokeWidth=0)

                 combination  count  n_sources
                        CLDR    416          1
CLDR + ISO 639-2 + Wikimedia    218          3
            CLDR + ISO 639-2    181          2
                   Wikimedia     28          1
            CLDR + Wikimedia     20          2
                   ISO 639-2     14          1
       ISO 639-2 + Wikimedia      3          2

Saved for RAW Graphs → /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/metadata_files/source_overlap_venn.csv


alt.LayerChart(...)

### CLDR version provenance and the v45 supplement

The `cldr_version` column records which CLDR release each row came from:

| `cldr_version` | Rows | Meaning |
|---|---|---|
| `'48.2.0'` | ~802 | Core CLDR 48.2 JSON fetch — authoritative for scripts, directionality, modern-use flag |
| `'45'` | 23 | CLDR v45 supplement — languages present in CLDR 45, dropped in 48.2 |
| `''` | ~55 | Wikimedia-only or LOC-only rows — no CLDR data at all |

**Why the supplement exists.** CLDR removes locales when their communities stop submitting Core data — font samples, collation rules, date formats — required for OS and software internationalisation. Between CLDR 45 and 48.2, 23 ISO 639-3 languages were removed on this basis (see the [CLDR 47 release notes](https://cldr.unicode.org/downloads/cldr-47) for the stated policy). That criterion does not apply here: a language can be fully translatable by an LLM without any community ever having submitted a CLDR locale file. All 23 are valid, living languages; the pipeline retains them marked `sources='cldr_v45'` so they appear in all downstream analysis alongside the rest.

Note: the source overlap chart above treats supplement codes as "CLDR only" (since their `sources` field contains the string `cldr`). The chart below isolates them explicitly.

In [7]:
if "cldr_version" not in df.columns:
    print("⚠  cldr_version not in CSV — re-run generate_language_codes.py to see this section.")
else:
    _ver_label = {
        "48.2.0": "CLDR 48.2 (primary)",
        "45":     "CLDR v45 supplement",
        "":       "Wikimedia / LOC only",
    }
    ver_counts = (
        df["cldr_version"].fillna("").map(_ver_label).fillna("other")
        .value_counts()
        .rename_axis("source")
        .reset_index(name="count")
    )
    print("Version provenance breakdown:")
    print(ver_counts.to_string(index=False))
    print()

    supp = (
        df[df["cldr_version"] == "45"]
        [["language_code", "language_name", "directionality", "family_name"]]
        .sort_values(["family_name", "language_code"])
        .reset_index(drop=True)
    )
    print(f"CLDR v45 supplement — {len(supp)} languages retained from CLDR 45:")
    print(supp.to_string(index=False))

    # ── Chart: supplement languages by family ────────────────────────────────
    supp_fam = supp["family_name"].value_counts().reset_index()
    supp_fam.columns = ["family", "count"]

    bars = alt.Chart(supp_fam).mark_bar(color="#c0392b").encode(
        y=alt.Y("family:N", sort="-x", title=None),
        x=alt.X("count:Q", title="languages in supplement"),
        tooltip=["family:N", "count:Q"],
    )
    text = bars.mark_text(align="left", dx=4, fontSize=10).encode(
        text="count:Q", color=alt.value("black"))

    (bars + text).properties(
        width=420, height=220,
        title=alt.Title(
            "CLDR v45 supplement: 23 languages by family",
            subtitle="Dropped from CLDR 48.2 for contributor-policy reasons, not linguistic ones — retained here for scholarly translation coverage",
        )
    )

Version provenance breakdown:
              source  count
 CLDR 48.2 (primary)    812
Wikimedia / LOC only     45
 CLDR v45 supplement     23

CLDR v45 supplement — 23 languages retained from CLDR 45:
language_code              language_name directionality                     family_name
          shu             Chadian Arabic            rtl          Afro-Asiatic languages
          lou           Louisiana Creole            ltr             Creoles and pidgins
          ike Eastern Canadian Inuktitut            ltr          Eskimo-Aleut languages
          gom               Goan Konkani            ltr         Indo-European languages
          hax             Southern Haida            ltr                Language isolate
          hdn             Northern Haida            ltr                Language isolate
          lsm                     Saamia            ltr     Niger-Kordofanian languages
          mye                      Myene            ltr     Niger-Kordofanian languages
       

### English Name Coverage: SIL ISO 639-3 Enrichment

CLDR's English locale-names file (`en/languages.json`) covers ~693 of its ~802 language codes. The remaining ~190 CLDR codes have no entry and fall through to `en_names.get(code, code)` in `parse_cldr_json.py` — leaving the ISO code itself as the `language_name`. Combined with a small number of codes from other sources that also lacked names, **216 of the 880 codes (~25%) had code-as-name** before enrichment.

These are not obscure codes: they include living languages like Abaza (`abq`), Bhilali (`bhi`), Kanauji (`bjj`), and Balanta-Ganja (`bjt`), as well as ancient and undeciphered scripts like Mycenaean Greek (`gmy`), Linear A (`lab`), Carian (`xcr`), and Meroitic (`xmr`).

`enrich_language_names_from_sil()` in `generate_language_codes.py` patches these rows using SIL's ISO 639-3 reference name table (~7,900 codes), which covers 214 of the 216. The remaining two are filled from a hardcoded `_SIL_NAME_OVERRIDES` constant:

| Code | Name | Reason not in SIL |
|------|------|-------------------|
| `kro` | Kru languages | ISO 639-5 group code; SIL ISO 639-3 covers individual languages only |
| `tokipona` | Toki Pona | Constructed language; Wikimedia-only composite tag with no ISO 639-3 assignment |

After enrichment, all 880 codes have a human-readable `language_name`. The table below shows the 216 previously unnamed codes and the names now assigned.

In [8]:
import json as _json

# Ground truth: which codes CLDR's en/languages.json directly names
_en_langs_path = os.path.join(
    DATA_DIR, "metadata_files", "cldr-cache",
    "cldr-localenames-full-48.2.0", "main", "en", "languages.json"
)
with open(_en_langs_path, encoding="utf-8") as _f:
    _cldr_en_names = set(
        _json.load(_f)["main"]["en"]["localeDisplayNames"]["languages"].keys()
    )

# SIL reference names
_sil_path = os.path.join(DATA_DIR, "metadata_files", "cldr-cache", "iso-639-3.tab")
_sil = pd.read_csv(_sil_path, sep="\t", dtype=str, na_filter=False)
_sil_names = dict(zip(_sil["Id"].str.strip(), _sil["Ref_Name"].str.strip()))
_sil_overrides = {"kro": "Kru languages", "tokipona": "Toki Pona"}

# Previously unnamed codes fall into two groups:
# 1. CLDR codes (sources contains 'cldr', not 'cldr_v45') absent from CLDR's en/languages.json
# 2. The two hardcoded overrides (kro: ISO 639-5 group code; tokipona: Wikimedia-only tag)
#    — tokipona is wikimedia_only so would otherwise slip through the CLDR filter
_unnamed_mask = (
    (
        df["sources"].str.contains("cldr", na=False)
        & ~df["sources"].str.fullmatch("cldr_v45", na=False)
        & ~df["language_code"].isin(_cldr_en_names)
    )
    | df["language_code"].isin(_sil_overrides)
)

def _fill_source(code):
    if code in _sil_overrides:
        return "Hardcoded override"
    if code in _sil_names:
        return "SIL ISO 639-3"
    return "Other"

_sil_filled = df[_unnamed_mask].copy()
_sil_filled["fill_source"] = _sil_filled["language_code"].map(_fill_source)

print(f"Codes with no CLDR English name (previously code-as-name): {len(_sil_filled)}")
print(f"  Filled by SIL ISO 639-3:  {(_sil_filled['fill_source'] == 'SIL ISO 639-3').sum()}")
print(f"  Filled by override:       {(_sil_filled['fill_source'] == 'Hardcoded override').sum()}")
print(f"  Still unnamed:            {(_sil_filled['fill_source'] == 'Other').sum()}")
print()

_sil_filled[["language_code", "language_name", "family_name", "modern_language", "fill_source"]].sort_values(
    ["family_name", "language_code"]
).reset_index(drop=True)

Codes with no CLDR English name (previously code-as-name): 216
  Filled by SIL ISO 639-3:  214
  Filled by override:       2
  Still unnamed:            0



,language_code,language_name,family_name,modern_language,fill_source
0,aii,Assyrian Neo-Aramaic,Afro-Asiatic languages,True,SIL ISO 639-3
1,apc,Levantine Arabic,Afro-Asiatic languages,True,SIL ISO 639-3
2,apd,Sudanese Arabic,Afro-Asiatic languages,True,SIL ISO 639-3
3,grr,Taznatit,Afro-Asiatic languages,True,SIL ISO 639-3
4,mey,Hassaniyya,Afro-Asiatic languages,True,SIL ISO 639-3
...,...,...,...,...,...
211,tts,Northeastern Thai,Tai-Kadai languages,True,SIL ISO 639-3
212,lab,Linear A,Undeciphered script,False,SIL ISO 639-3
213,kca,Khanty,Uralic languages,True,SIL ISO 639-3
214,mns,Mansi,Uralic languages,True,SIL ISO 639-3


### The `nan` edge case: Min Nan Chinese

**Min Nan Chinese** (`nan`) is ISO 639-3 code `nan` — the same three characters Python uses for *Not a Number*. When `pd.read_csv` encounters the string `"nan"` in a column with no explicit converter, it silently promotes it to `float('nan')`, erasing the code. The language is in the dataset and its family is correctly assigned, but any code-keyed lookup silently misses it without the fix.

**Why the pipeline is safe.** `load_language_codes()` reads the CSV with `converters={'language_code': str}`, which forces `"nan"` to stay a string. Any code that reads this CSV without that converter is exposed to the silent drop.

**How `_family_source` handles it.** The function detects `pd.isna(raw_code)` and normalises to the string `"nan"` before any lookup. `nan` is now in `MANUAL_LANG_TO_SET5` (mapped to `'zhx'` — Sinitic languages → Sino-Tibetan), so it correctly returns **"Manual mapping"** and the "Unassigned" count is zero.

### Wikimedia-only language codes

Languages that appear in Wikimedia but not CLDR fall into four distinct classes:

- **Composite codes Wikimedia invented** (`bat-smg`, `be-x-old`, `fiu-vro`, `map-bms`, `nds-nl`, `roa-rup`, `zh-classical`, `zh-min-nan`, `zh-tw`, `zh-yue`, `uz_AF`): community-assigned `xx-yyy` or `xx_YY` tags with no ISO equivalent. These represent writing communities that achieved Wikipedia scale but were never submitted to ISO for standardisation.
- **ISO 639-3-only languages below CLDR's threshold**: codes like `bcl`, `bxr`, `cdo`, `diq`, `mrh`, `nrm` have ISO 639-3 entries but CLDR only tracks languages used in OS/software localisation — a narrower criterion than "has a Wikipedia."
- **Contested, collective, or deprecated ISO status**: `mo` (Moldovan, ISO 639-1 code withdrawn — treated as Romanian), `nah` (Nahuatl, an ISO 639-2 *collective* cover code for the whole language family), `crn` (Montenegrin, ISO 639-3 `cnr` only assigned 2017), and `als` (a code collision — ISO 639-3 `als` is Tosk Albanian, not Alemannic German, which is `gsw`).
- **Constructed language**: `tokipona` (Toki Pona, created 2001); ISO 639-3 code `tok` was only assigned in 2022.

The `crn`/Montenegrin case is the sharpest failure mode in the dataset: the model family-assignment used only the bare code, inferred Woods Cree (for whom ISO 639-3 `crn` *is* the correct assignment), and produced a row saying Montenegrin belongs to the Indigenous North American family. The error was only discoverable because the pipeline logs the rationale alongside each assignment. It is corrected in `MANUAL_LANG_TO_SET5` and `language_family_assignments.json`.

## 1.2 Constructing the Target Set

Why 880? Four candidate target sets range from the most conservative (Wikimedia-only: 269 languages with active Wikipedia projects) to the broadest (all 880 codes). Each filter represents a different answer to: *which languages should we ask the pipeline to translate?*

The pipeline uses **all 880** — including languages without Wikipedia presence, without ISO 639-2 codes, and including languages CLDR marks as "non-modern." The sections below justify each step of that expansion.

In [9]:
filters = {
    f"All {n_languages} (pipeline default)":  pd.Series([True] * len(df)),
    "Wikimedia only":                          df["in_wikimedia"].fillna(False),
    "Modern + ISO 639-2 or Wikimedia":         df["modern_language"] & (df["in_iso639_2"] | df["in_wikimedia"]),
    "All modern":                              df["modern_language"].fillna(False),
}

rows = []
for label, mask in filters.items():
    subset = df[mask]
    rows.append({
        "filter": label,
        "total":  len(subset),
        "ltr":    (subset["directionality"] == "ltr").sum(),
        "rtl":    (subset["directionality"] == "rtl").sum(),
        "iso639_1": subset["in_iso639_1"].sum(),
        "iso639_2": subset["in_iso639_2"].sum(),
        "wikimedia": subset["in_wikimedia"].sum(),
    })
filter_df = pd.DataFrame(rows)
filter_df

,filter,total,ltr,rtl,iso639_1,iso639_2,wikimedia
0,All 880 (pipeline default),880,802,78,187,416,269
1,Wikimedia only,269,253,16,185,221,269
2,Modern + ISO 639-2 or Wikimedia,452,428,24,186,404,267
3,All modern,855,787,68,186,404,267


In [10]:
filter_order = filter_df["filter"].tolist()

bar = alt.Chart(filter_df).mark_bar().encode(
    y=alt.Y("filter:N", sort=filter_order, title=None),
    x=alt.X("total:Q", title="languages in target set"),
    color=alt.Color("filter:N", legend=None, scale=alt.Scale(scheme="tableau10")),
    tooltip=["filter:N", "total:Q", "ltr:Q", "rtl:Q", "wikimedia:Q"],
).properties(width=420, height=160, title="Target set size by filter strategy")

text = bar.mark_text(align="left", dx=4, fontSize=10).encode(
    text="total:Q", color=alt.value("black"))

bar + text

alt.LayerChart(...)

In [11]:
# Stacked LTR/RTL breakdown
dir_rows = []
for _, r in filter_df.iterrows():
    dir_rows.append({"filter": r["filter"], "direction": "LTR", "count": r["ltr"]})
    dir_rows.append({"filter": r["filter"], "direction": "RTL", "count": r["rtl"]})
dir_df = pd.DataFrame(dir_rows)

dir_bar = alt.Chart(dir_df).mark_bar().encode(
    y=alt.Y("filter:N", sort=filter_order, title=None),
    x=alt.X("count:Q", title="languages"),
    color=alt.Color("direction:N",
        scale=alt.Scale(domain=["LTR", "RTL"], range=["#1976d2", "#e64a19"]),
        title="direction"),
    tooltip=["filter:N", "direction:N", "count:Q"],
).properties(width=420, height=160, title="LTR / RTL split by filter strategy")

dir_bar

alt.Chart(...)

### Expansion Steps

Moving from Wikimedia-only (269) to the full 880 adds two further tranches: languages in ISO 639-2 or Wikimedia (452 total, +183), then all CLDR-modern languages (855 total, +403), then the 25 non-modern languages discussed below.

In [12]:
wiki_mask  = df["in_wikimedia"].fillna(False)
iso2_mask  = df["modern_language"] & (df["in_iso639_2"] | df["in_wikimedia"])
modern_mask = df["modern_language"].fillna(False)

# Languages in ISO2-or-Wikimedia but NOT in Wikimedia-only
iso2_additions = df[iso2_mask & ~wiki_mask][
    ["language_code", "language_name", "directionality", "family_name", "sources"]
].sort_values("language_name").reset_index(drop=True)

# Languages in All-Modern but NOT in ISO2-or-Wikimedia
modern_additions = df[modern_mask & ~iso2_mask][
    ["language_code", "language_name", "directionality", "family_name", "sources"]
].sort_values("language_name").reset_index(drop=True)

print(f"Added by expanding to ISO 639-2 or Wikimedia: {len(iso2_additions)}")
print(f"Added by expanding to all modern: {len(modern_additions)}")
print()

print("=== ISO 639-2 expansion additions (sample) ===")
print(iso2_additions.head(20).to_string(index=False))

Added by expanding to ISO 639-2 or Wikimedia: 185
Added by expanding to all modern: 403

=== ISO 639-2 expansion additions (sample) ===
language_code language_name directionality                     family_name       sources
          ace      Acehnese            ltr          Austronesian languages cldr|iso639_2
          ach         Acoli            ltr          Nilo-Saharan languages cldr|iso639_2
          ada       Adangme            ltr     Niger-Kordofanian languages cldr|iso639_2
          ady        Adyghe            ltr             Caucasian languages cldr|iso639_2
          afh      Afrihili            ltr            Artificial languages      loc_only
          ain          Ainu            ltr                Language isolate cldr|iso639_2
          ale         Aleut            ltr          Eskimo-Aleut languages cldr|iso639_2
          grc Ancient Greek            ltr         Indo-European languages cldr|iso639_2
          anp        Angika            ltr         Indo-Europea

In [13]:
print("=== Additional languages from CLDR-only (all modern, no ISO 639-2 or Wikimedia) ===")
print(modern_additions.to_string(index=False))

=== Additional languages from CLDR-only (all modern, no ISO 639-2 or Wikimedia) ===
language_code                  language_name directionality                       family_name  sources
          abq                          Abaza            ltr               Caucasian languages     cldr
          abr                          Abron            ltr       Niger-Kordofanian languages     cldr
          agq                          Aghem            ltr       Niger-Kordofanian languages     cldr
          bss                         Akoose            ltr       Niger-Kordofanian languages     cldr
          akz                        Alabama            ltr   North American Indian languages     cldr
          arq                Algerian Arabic            rtl            Afro-Asiatic languages     cldr
          ase         American Sign Language            ltr                    Sign languages cldr_v45
          amo                            Amo            ltr       Niger-Kordofanian language

### Non-Modern Languages

CLDR (Unicode's Common Locale Data Repository) includes a type flag that distinguishes languages it considers "modern" from those it labels `'O'` — a catch-all for ancient, extinct, and constructed languages. The `modern_language` column in `language_codes_comprehensive.csv` is derived directly from this flag (`modern_language = (cldr_type != 'O')`). It reflects CLDR's purpose — building locale data for commercial software internationalization — not a linguistic or scholarly judgment about which languages matter.

**We do not filter on it.** The pipeline includes all 880 languages, and these 25 appear in every downstream analysis alongside the rest. Two of them have active Wikipedia projects and one has an ISO 639-1 code — the same markers of real-world use we apply to any other language. The fact that an LLM attempts or refuses to translate into Latin, Linear A, or Esperanto is itself a finding worth keeping in the data.

The table below is a record of which languages CLDR chose to flag, not a list of languages the pipeline excludes.

In [14]:
non_modern = df[~df["modern_language"].fillna(True)].copy()
non_modern["has_wikipedia"] = non_modern["in_wikimedia"].fillna(False)
non_modern["has_iso639_1"]  = non_modern["in_iso639_1"].fillna(False)
non_modern["has_iso639_2"]  = non_modern["in_iso639_2"].fillna(False)

display_cols = ["language_code", "language_name", "directionality",
                "has_wikipedia", "has_iso639_1", "has_iso639_2", "family_name"]

print(f"Total non-modern: {len(non_modern)}")
print(f"  With active Wikipedia: {non_modern['has_wikipedia'].sum()}")
print(f"  With ISO 639-1 code:   {non_modern['has_iso639_1'].sum()}")
print()
non_modern[display_cols].sort_values(
    ["has_wikipedia", "has_iso639_1"], ascending=False
).reset_index(drop=True)

Total non-modern: 25
  With active Wikipedia: 2
  With ISO 639-1 code:   1



,language_code,language_name,directionality,has_wikipedia,has_iso639_1,has_iso639_2,family_name
0,arc,Aramaic,rtl,True,False,True,Afro-Asiatic languages
1,got,Gothic,ltr,True,False,True,Indo-European languages
2,ae,Avestan,rtl,False,True,True,Indo-European languages
3,akk,Akkadian,ltr,False,False,True,Afro-Asiatic languages
4,ecy,Eteocypriot,rtl,False,False,False,Indo-European languages
5,egy,Ancient Egyptian,ltr,False,False,True,Afro-Asiatic languages
6,gmy,Mycenaean Greek,rtl,False,False,False,Indo-European languages
7,hit,Hittite,ltr,False,False,True,Indo-European languages
8,lab,Linear A,ltr,False,False,False,Undeciphered script
9,mnc,Manchu,ltr,False,False,True,Altaic languages


### A structural edge case: `uz_AF`

`uz_AF` (Afghan Uzbek) is a Wikimedia-only code for the variety spoken in northern Afghanistan, which retains Perso-Arabic script — Wikimedia correctly marks it `rtl`. But `uz_AF` has no CLDR script data, so the script-based pass defaults to `ltr`.

The pipeline corrects this with a fallback in `build_comprehensive`: for any Wikimedia-only code with no CLDR script data, if `directionality_wikimedia == 'rtl'`, restore `rtl`. This is the one case in the dataset where the less institutionally curated source (Wikimedia) is closer to empirical truth than the CLDR-derived default.

## 1.3 Language Families

Family membership is the primary grouping variable for downstream analysis. The three sub-sections here document how families distribute across filter strategies, how large individual families are, and where each family assignment came from.

In [15]:
family_rows = []
for label, mask in filters.items():
    subset = df[mask]
    counts = subset["family_name"].fillna("Unknown / unassigned").value_counts().reset_index()
    counts.columns = ["family", "count"]
    counts["filter"] = label
    family_rows.append(counts)
family_long = pd.concat(family_rows, ignore_index=True)

# Focus on top 15 families across all filters
top_families = (
    family_long.groupby("family")["count"].sum()
    .nlargest(15).index.tolist()
)
plot_df = family_long[family_long["family"].isin(top_families)]

alt.Chart(plot_df).mark_bar().encode(
    x=alt.X("count:Q", title="languages"),
    y=alt.Y("family:N", sort="-x", title=None),
    color=alt.Color("filter:N", sort=list(filters.keys()),
        scale=alt.Scale(scheme="tableau10"), title="filter"),
    xOffset="filter:N",
    tooltip=["filter:N", "family:N", "count:Q"],
).properties(
    width=500, height=420,
    title="Top 15 language families by filter strategy",
)

alt.Chart(...)

### Family Size Distribution

Family size varies from Indo-European (243) to Chukotko-Kamchatkan (2), Sign languages (1), and Australian languages (1). This matters for interpreting downstream results: pattern claims about very small families rest on few data points and should be read as illustrative rather than statistically grounded.

In [16]:
family_sizes = (
    df["family_name"].fillna("Unassigned")
    .value_counts()
    .rename_axis("family")
    .reset_index(name="n_languages")
    .sort_values("n_languages", ascending=False)
    .reset_index(drop=True)
)

print(f"Total distinct families: {len(family_sizes)}")
print()
print(family_sizes.to_string(index=False))

bar = alt.Chart(family_sizes).mark_bar(color="#1976d2").encode(
    y=alt.Y("family:N",
            sort=alt.EncodingSortField("n_languages", order="descending"),
            title=None),
    x=alt.X("n_languages:Q", title="languages in family"),
    tooltip=["family:N", "n_languages:Q"],
).properties(
    width=420,
    height=max(240, len(family_sizes) * 14),
    title=f"Language family sizes (all {n_languages} languages)",
)
bar

Total distinct families: 29

                           family  n_languages
          Indo-European languages          244
      Niger-Kordofanian languages          158
           Austronesian languages           84
  North American Indian languages           65
           Afro-Asiatic languages           51
           Sino-Tibetan languages           51
                 Altaic languages           34
           Nilo-Saharan languages           29
                 Uralic languages           27
              Creoles and pidgins           17
              Caucasian languages           15
             Artificial languages           15
  South American Indian languages           14
         Austro-Asiatic languages           14
                 Language isolate           14
              Dravidian languages           10
              Tai-Kadai languages           10
Central American Indian languages            7
           Eskimo-Aleut languages            5
             Hmong-Mien languag

alt.Chart(...)

### Family Name Assignment

Where did each language's `family_name` come from?

- **Manual mapping** — hardcoded in `MANUAL_LANG_TO_SET5` in `generate_language_codes.py`. Covers major ISO 639-1 and Wikimedia codes, plus 23 CLDR-only codes added when moving from the HTML scrape (CLDR 45) to the JSON source (CLDR 48.2).
- **JSON assessment** — `language_family_assignments.json`. Claude-generated linguistic inference for codes not covered by the manual mapping; each entry includes a rationale.

Family hierarchy (the `iso639_5_family` walk-up that determines which top-level family a sub-group belongs to) comes from CLDR's `languageGroups.json` — the same `cldr-core` npm package used for scripts and directionality. For subfamily codes used in `MANUAL_LANG_TO_SET5` that are not in CLDR's own hierarchy (e.g. `hyx` for Armenian, `sqj` for Albanian), names are drawn from a hardcoded `_ISO639_5_NAMES` dict in `parse_cldr_json.py`.

The stacked bars below show what proportion of each family's codes were assigned via each method.

In [17]:
source_order = ["Manual mapping", "JSON assessment", "Unassigned"]
source_colors = {
    "Manual mapping":  "#1976d2",
    "JSON assessment": "#e64a19",
    "Unassigned":      "#9e9e9e",
}

source_counts = (
    df["family_name_source"].value_counts()
    .reindex(source_order, fill_value=0)
    .reset_index()
)
source_counts.columns = ["source", "count"]
source_counts["pct"] = (source_counts["count"] / len(df) * 100).round(1)
source_counts["label"] = source_counts["count"].astype(str) + " (" + source_counts["pct"].astype(str) + "%)"

# ── Bar: overall source breakdown ────────────────────────────────────────────
bar = alt.Chart(source_counts).mark_bar().encode(
    y=alt.Y("source:N", sort=source_order, title=None),
    x=alt.X("count:Q", title="number of language codes"),
    color=alt.Color(
        "source:N",
        sort=source_order,
        scale=alt.Scale(
            domain=list(source_colors.keys()),
            range=list(source_colors.values()),
        ),
        legend=None,
    ),
    tooltip=["source:N", "count:Q", "pct:Q"],
).properties(width=380, height=100, title=f"Family name assignment source (all {n_languages} codes)")

text = bar.mark_text(align="left", dx=5, fontSize=10).encode(
    text="label:N", color=alt.value("black"))

# ── Stacked bar: source mix per top family ────────────────────────────────────
top_fams = (
    df[df["family_name"].notna() & (df["family_name"] != "")]
    ["family_name"].value_counts().nlargest(14).index.tolist()
)
fam_src = (
    df[df["family_name"].isin(top_fams)]
    .groupby(["family_name", "family_name_source"])
    .size()
    .reset_index(name="count")
)
fam_order = (
    fam_src.groupby("family_name")["count"].sum()
    .sort_values(ascending=False).index.tolist()
)

stacked = alt.Chart(fam_src).mark_bar().encode(
    y=alt.Y("family_name:N", sort=fam_order, title=None),
    x=alt.X("count:Q", title="languages", stack="normalize",
            axis=alt.Axis(format="%")),
    color=alt.Color(
        "family_name_source:N",
        sort=source_order,
        scale=alt.Scale(
            domain=list(source_colors.keys()),
            range=list(source_colors.values()),
        ),
        title="source",
    ),
    tooltip=["family_name:N", "family_name_source:N", "count:Q"],
).properties(width=380, height=360, title="Source mix by family (top 14, normalised)")

(bar + text) & stacked

alt.VConcatChart(...)

## 1.4 Script and Directionality

Script type and writing direction are structural features that propagate through the entire pipeline — affecting translation display, search term handling, and quality-flag logic. The sub-sections here document the script landscape, the overall LTR/RTL split, and how directionality distributes across families.

In [18]:
script_rows = []
for label, mask in filters.items():
    subset = df[mask]
    counts = subset["primary_script"].fillna("Unknown").value_counts().reset_index()
    counts.columns = ["script", "count"]
    counts["filter"] = label
    script_rows.append(counts)
script_long = pd.concat(script_rows, ignore_index=True)

top_scripts = (
    script_long.groupby("script")["count"].sum()
    .nlargest(12).index.tolist()
)
script_plot = script_long[script_long["script"].isin(top_scripts)]

alt.Chart(script_plot).mark_bar().encode(
    x=alt.X("count:Q", title="languages"),
    y=alt.Y("script:N", sort="-x", title=None),
    color=alt.Color("filter:N", sort=list(filters.keys()),
        scale=alt.Scale(scheme="tableau10"), title="filter"),
    xOffset="filter:N",
    tooltip=["filter:N", "script:N", "count:Q"],
).properties(
    width=500, height=380,
    title="Top 12 scripts by filter strategy",
)

alt.Chart(...)

### LTR / RTL Summary

802 languages (91.1%) are left-to-right; 78 (8.9%) are right-to-left. RTL languages are concentrated in Afro-Asiatic (Arabic-family and Semitic languages) and a small number of historical scripts.

In [19]:
df.directionality.value_counts()

directionality
ltr    802
rtl     78
Name: count, dtype: int64

### Directionality by Family and Source

For each family: how many languages are LTR vs RTL, and which source database brought them in? The two panels use independent x-scales because RTL languages are far fewer than LTR in every family.

In [20]:
dir_rows = []
for (family, direction), grp in df.groupby(["family_name", "directionality"]):
    dir_rows.append({
        "family":    family,
        "direction": direction.upper(),
        "count":     len(grp),
    })
dir_df = pd.DataFrame(dir_rows)

fam_order = (
    dir_df.groupby("family")["count"].sum()
    .sort_values(ascending=False).index.tolist()
)

alt.Chart(dir_df).mark_bar().encode(
    y=alt.Y("family:N", sort=fam_order, title=None),
    x=alt.X("count:Q", title="number of languages"),
    color=alt.Color(
        "direction:N",
        scale=alt.Scale(domain=["LTR", "RTL"], range=["#1976d2", "#e64a19"]),
        title="Direction",
    ),
    tooltip=["family:N", "direction:N", "count:Q"],
).facet(
    column=alt.Column(
        "direction:N", sort=["LTR", "RTL"], title=None,
        header=alt.Header(labelFontWeight="bold", labelFontSize=13),
    ),
).resolve_scale(x="independent").properties(
    title=alt.Title(
        "Directionality by language family",
        subtitle="Independent x-scales — RTL languages are far fewer in every family",
    )
)

alt.FacetChart(...)